#### NCEP GFS Forecasts (1.00 degree grid) Data(atmosphere)을  JSON Format File 로 변환을 위한 분석 Note
###### NCEP NOMADS Version 2.3.29, November 2026


- GRIB2 (General Regularly-distributed Information in Binary Form, Edition 2)는 세계기상기구(WMO)에서 제정한 수치예보 모델 및 기상·해양 격자 자료(Gridded Data) 전송·저장을 위한 표준 바이너리 포맷입니다.
- 기존 GRIB1의 한계를 극복하고 확장성과 유연성을 높이기 위해 템플릿(Template) 기반의 구조로 설계되었습니다.
- GRIB2 파일은 하나 이상의 GRIB 메시지(Message)가 연속된 비트 스트림(Bit-stream) 형태로 구성되어 있으며, 각 메시지는 여러 개의 섹션(Section)으로 나뉩니다

1. GRIB2 메시지의 전체 구조 (Section Architecture)
- 하나의 GRIB2 메시지는 필수 및 선택 섹션들을 포함하여 총 8개의 섹션(Section 0 ~ 7)과 마지막 종료 표시자로 구성됩니다.


| 섹션 번호 | 섹션 이름 (English) | 주요 역할 및 설명 |
| :---: | :--- | :--- |
| **Section 0** | **Indicator Section** | 메시지의 시작(`GRIB` 문자열), GRIB Edition 번호(2), 전체 메시지의 총 바이트 길이 등 기본 정보를 담습니다. |
| **Section 1** | **Identification Section** | 자료를 생성한 센터(Originating Center/Sub-center), 마스터/로컬 테이블 버전, 모델 기준 시간(Reference Time), 자료의 상태(분석, 예보 등)를 정의합니다. |
| **Section 2** | **Local Use Section** | **(선택적)** 발행 기관이나 모델 고유의 로컬 정보를 담기 위한 공간입니다. 없을 수도 있습니다. |
| **Section 3** | **Grid Definition Section** | 격자 형태(좌표계)를 정의합니다. 위경도 격자(Lat/Lon), 회전 위경도, 메르카토르(Mercator), 램버트 정각 원추(Lambert Conformal), 극스테레오그래픽 등 격자 구조와 투영법 정보를 담습니다. |
| **Section 4** | **Product Definition Section** | 어떤 기상 변수인지 정의합니다. **Discipline(분야), Category(기상 범주), Parameter Number(변수 번호)**의 삼중항(Triplet)을 사용하며, 생성 시간, 예보 시간, 통계 처리 방식(평균, 누적 등)을 템플릿을 통해 상세히 규정합니다. |
| **Section 5** | **Data Representation Section** | 실제 격자점 값들을 압축하거나 인코딩하기 위해 사용된 수학적 방법(단순 포장, 복합 포장, 스펙트럼 데이터 압축, JPEG 2000, PNG 등)과 비트 할당 방식을 정의합니다. |
| **Section 6** | **Bitmap Section** | 결측치(Missing value)가 있는 격자점을 처리하기 위한 비트맵을 정의합니다. (모든 값이 유효할 경우 생략되거나 간소화됨) |
| **Section 7** | **Data Section** | Section 5와 6에서 정의한 방식에 따라 실제 압축·패킹(Packed)된 수치 데이터 비트열이 담기는 가장 큰 영역입니다. |
| **End** | **End Section** | 메시지의 끝을 알리는 ASCII 문자열 **`7777`**로 구성됩니다. |


2. GRIB2 핵심 특징 및 개선 사항
- 템플릿 및 테이블 기반 (Template-driven): 고정된 코드가 아닌 표준 테이블과 템플릿 구조를 사용하여 새로운 변수, 새로운 격자 체계, 새로운 압축 알고리즘이 나와도 기존 시스템의 대규모 수정 없이 유연하게 확장할 수 있습니다.
- 변수 체계 (Triplet): GRIB1에서는 변수 코드가 단순 번호였으나, GRIB2는 (Discipline, Category, Parameter) 형태의 삼중항 체계를 사용하여 기상, 해양, land, 화학 대기 등 다양한 분야의 방대한 변수를 체계적으로 분류합니다. (예: 기온, 해수면 온도, 풍속 등)
- 고정밀 좌표: 위도와 경도를 밀도가 더 높은 마이크로도($10^{-6}$도) 단위까지 표현할 수 있어 정밀도가 향상되었습니다.
- 고효율 압축: GRIB1 대비 JPEG 2000이나 PNG 등의 진보된 압축 알고리즘을 지원하여 대용량 수치예보 자료의 파일 용량을 크게 줄였습니다.

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import sqlite3
import json
import os
import glob
from pathlib import Path

In [ ]:
file_path = "/Users/unchil/AndroidStudioProjects/earth/public/data/download/gfs.t18z.pgrb2.1p00.f000"
target_dir = Path("./data/download/")

In [ ]:
ds = xr.open_dataset(
    file_path,
    engine="cfgrib",
    backend_kwargs={
        "filter_by_keys": {"typeOfLevel": "isobaricInhPa", "level": 1000.},
        "indexpath": "",
    },
)
ds.info()

In [ ]:
refTime = pd.to_datetime(ds["time"].values ).strftime('%Y-%m-%dT%H:%M:%S.000Z')
validTime = pd.to_datetime(ds["valid_time"].values ).strftime('%Y%m%d%H')
forecastTime = int(f"{int(ds['step'].values / np.timedelta64(1, 'h')):02d}")

refTime, validTime, forecastTime

In [ ]:
centerName = ds.attrs['GRIB_centreDescription']
refTime = pd.to_datetime(ds["time"].values ).strftime('%Y-%m-%dT%H:%M:%S.000Z')
validTime = pd.to_datetime(ds["valid_time"].values ).strftime('%Y-%m-%dT%H:%M:%S.000Z')
forecastTime = int(f"{int(ds['step'].values / np.timedelta64(1, 'h')):02d}")
surface1Value = float(ds["isobaricInhPa"].item()) * 100

shape = ds.data_vars['t'].shape
numberPoints = ds.data_vars['t'].values.flatten().size
la2 = float(ds['t'].latitude.values.min())
la1 = float(ds['t'].latitude.values.max())
lo1 = float(ds['t'].longitude.values.min())
lo2 = float(ds['t'].longitude.values.max())
data = [ round(float(x), 5) for x in ds['t'].values.flatten() ]

In [ ]:
payload = [{"header":{"center":7,"centerName":centerName, "refTime":refTime, "forecastTime":forecastTime, "surface1Value":surface1Value,"numberPoints":numberPoints,"winds":"true","nx":shape[1],"ny":shape[0],"basicAngle":0,"lo1":lo1,"la1":la1,"lo2":lo2,"la2":la2,"dx":1.0,"dy":1.0}, "data":data}]

In [ ]:
tupleRow = ('2026111406', 'air', 'temperature', 'isobaric', '1000hPa', json.dumps(payload))

In [ ]:
conn = sqlite3.connect('/Users/unchil/AndroidStudioProjects/OceanWaterInfo/oceanwater.sqlite')
cursor = conn.cursor()
cursor.execute( "INSERT INTO Earth_Current_Tbl (forecast_time, mode, type, surface, level, grib2json ) VALUES (?, ?, ?, ?, ?, ?) ",  tupleRow)
conn.commit()

#### parameter 정의
- **특수 키워드 `"wind"`** :
    - `parameterNumber: 2, 3` :
         - 바람 벡터 연산에 필요한 `UGRD`(동서 성분, parameterCategory 2 - parameterNumber 2)와 `VGRD`(남북 성분, parameterCategory 2 - parameterNumber 3)를 한 번에 동시 추출합니다.


- **주요 기상 변수 파라미터 코드 매핑**:

| parameterCategory| parameterCategoryName | parameterNumber | parameterNumberName | 설명 | `earth` 레이어 매핑 |
|:---|:---|:---|:---|:---|:---|
| **0**|**Temperature** | `0` | `TMP` | 기온 (Temperature, K) | 기온 (`temp`) |
| **0**|**Temperature** | `7` | `DPT` | 이슬점 온도 (Dew Point, K) | 이슬점 |
| **1**|**Moisture** | `1` | `RH` | 상대습도 (Relative Humidity, %) | 상대습도 (`relative_humidity`) |
| **1**|**Moisture** | `54` | `PWAT` | 가강수량 (Precipitable Water, $\text{kg/m}^2$) | 총 가강수량 (`total_precipitable_water`) |
| **2**|**Momentum** | `2`/`3` (wind) | `UGRD` & `VGRD` | U/V 풍속 벡터 성분 (m/s) | 바람 벡터 (`wind`) |
| **3**|**Mass** | `1` / `192` | `PRMSL` | 해면기압 (Pressure at MSL, Pa) | 해면기압 (`mean_sea_level_pressure`) |
| **6**|**Cloud** | `6` | `CWAT` | 구름수량 (Cloud Water, $\text{kg/m}^2$) | 총 구름수량 (`total_cloud_water`) |
| **7**|**Stability** | `6` | `CAPE` | 대기불안정도 (CAPE, J/kg) | 대기 대류 유효 전위 에너지 |




#### surface1Type 정의
- ex) "surface1Type":100,"surface1TypeName":"Isobaric surface","surface1Value":100000.0
- **surface1Value**
    - 지상~대류권 하부 (4개 층). 1000 hPa(약 100m), 925 hPa(약 800m), 850 hPa(약 1.5km), 700 hPa(약 3km)로 구성되어 하층 수증기 유입과 대기 불안정을 파악합니다.
    - 대류권 중·상부 (5개 층). 500 hPa(약 5.5km), 400 hPa(약 7.2km), 300 hPa(약 9.2km), 250 hPa(약 10.4km), 200 hPa(약 11.8km)로 구성되어 기압골, 기압능, 제트기류를 분석합니다.
    - 성층권 (7개 층). 150 hPa, 100 hPa(약 16.2km), 70 hPa, 50 hPa, 30 hPa, 20 hPa, 10 hPa(약 30km 이상)까지 이어지며 상층 대기 순환을 감시합니다.

In [ ]:
from pathlib import Path

In [ ]:
conn = sqlite3.connect('/Users/unchil/AndroidStudioProjects/OceanWaterInfo/oceanwater.sqlite')
cursor = conn.cursor()


oscar_dir = Path("../data/oscar/")
for json_file in oscar_dir.glob("*.json"):
    if os.path.exists(json_file):

        filename = "../data/oscar/2026111600-surface-currents-oscar-1.0.json"
        file_name_only = Path(json_file).name
        time_str = file_name_only.split("-")[0]


        with open(json_file, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
                tupleRow = (time_str, 'ocean', 'currents', 'surface', 'level', json.dumps(data))
                cursor.execute( "INSERT OR REPLACE INTO Earth_Tbl (forecast_time, mode, type, surface, level, grib2json ) VALUES (?, ?, ?, ?, ?, ?) ",  tupleRow)
                conn.commit()
            except json.JSONDecodeError as err:
                print(f"  [오류] JSON Data Insert ({json_file}): {err}")


In [ ]:
conn.close()